# Walkthrough: Embedding Choices Under a Fixed GIN Backbone

Paper: **How Embeddings Shape Graph Neural Networks: Classical vs Quantum-Oriented Node Representations** (arXiv:2604.15273v1)

> "This work studies, under a unified protocol, how different embedding modules influence downstream graph-level prediction when the GNN backbone is kept fixed."
>
> — §I. INTRODUCTION

This notebook uses tiny toy tensors so everything runs quickly on CPU. The structure matches the paper; only dimensions are reduced for demonstration.

In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.model import ModelConfig, GraphEmbeddingBenchmarkModel
from src.loss import GraphClassificationLoss

torch.manual_seed(7)

# Tiny CPU-friendly walkthrough dimensions
BATCH_SIZE = 2
NODES = 6
INPUT_DIM = 10
NUM_CLASSES = 3


## Configuration (§A. Experimental Setup)

> "Node embedding dim d = 32 (all methods)."
>
> "GIN (3 layers), hidden dim 64 ... dropout 0.2."
>
> — Table I

We keep the architecture pattern but use small toy dimensions to make checks instant.

In [ ]:
toy_config = ModelConfig(
    embedding_dim=16,
    gin_layers=2,
    gin_hidden_dim=24,
    mlp_head_hidden_dim=16,
    dropout=0.2,
    embedding_kind='qwalkvec_trainable',
    qwalk_steps=8,
    num_classes=NUM_CLASSES,
)
toy_config

## QWalkVec Component (Algorithm 2)

> "QWalkVec constructs node embeddings by simulating a coined quantum walk and recording how node visitation probabilities evolve over time."
>
> — §II.B, Algorithm 2

**Inline code excerpt (from `src/model.py`)**

In [ ]:
class NotebookQWalkVecEmbedding(nn.Module):
    """Algorithm 2 sketch used in src/model.py."""

    def __init__(self, steps: int, wp: float, wq: float, out_dim: int):
        super().__init__()
        self.steps = steps
        self.wp = wp
        self.wq = wq
        self.proj = nn.Linear(steps, out_dim)

    def forward(self, adj: torch.Tensor, node_mask: torch.Tensor) -> torch.Tensor:
        batch, n, _ = adj.shape
        descriptors = torch.zeros(batch, n, self.steps, dtype=adj.dtype, device=adj.device)
        for b in range(batch):
            n_valid = int(node_mask[b].sum().item())
            if n_valid == 0:
                continue
            local_adj = adj[b, :n_valid, :n_valid]  # (n_v, n_v)
            identity = torch.eye(n_valid, device=adj.device, dtype=adj.dtype)  # (n_v, n_v)
            transition = local_adj + identity  # (n_v, n_v)
            transition = transition / transition.sum(dim=-1, keepdim=True).clamp_min(1e-8)  # (n_v, n_v)
            walk_op = torch.softmax(self.wp * identity + self.wq * transition, dim=-1)  # (n_v, n_v)
            current = identity
            for t in range(self.steps):
                current = current @ walk_op  # (n_v, n_v)
                descriptors[b, :n_valid, t] = torch.diag(current)  # (n_v,)
        return self.proj(descriptors)  # (batch, n, steps) -> (batch, n, out_dim)


In [ ]:
qwalk = NotebookQWalkVecEmbedding(steps=8, wp=0.5, wq=4.0, out_dim=16)
adj = torch.randint(0, 2, (BATCH_SIZE, NODES, NODES), dtype=torch.float32)
adj = torch.triu(adj, diagonal=1)
adj = adj + adj.transpose(-1, -2)
mask = torch.ones(BATCH_SIZE, NODES, dtype=torch.bool)
z = qwalk(adj=adj, node_mask=mask)
assert z.shape == (BATCH_SIZE, NODES, 16), f'Unexpected shape: {z.shape}'
print(f'✓ QWalkVec embedding: {z.shape}')

## GIN Backbone (§B. Graph classifier and training protocol)

> "All embedding variants are evaluated with the same downstream classifier fψ implemented as a GIN backbone."
>
> — §B. Graph classifier and training protocol

The full model composes embedding stage + shared GIN + graph head.

In [ ]:
model = GraphEmbeddingBenchmarkModel(config=toy_config, input_dim=INPUT_DIM)
node_features = torch.randn(BATCH_SIZE, NODES, INPUT_DIM)
node_mask = torch.ones(BATCH_SIZE, NODES, dtype=torch.bool)
logits = model(node_features=node_features, adj=adj, node_mask=node_mask)
assert logits.shape == (BATCH_SIZE, NUM_CLASSES), f'Unexpected logits shape: {logits.shape}'
print(f'✓ Full model forward: {node_features.shape} -> {logits.shape}')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Loss Function (§B)

> "... parameters are optimized by minimizing cross-entropy on the training split ..."
>
> — §B. Graph classifier and training protocol

In [ ]:
loss_fn = GraphClassificationLoss()
targets = torch.tensor([0, 2], dtype=torch.long)
loss = loss_fn(logits=logits, targets=targets)
assert loss.ndim == 0, f'Expected scalar loss, got shape {loss.shape}'
print(f'✓ Loss computed: {loss.item():.4f}')

## One Training Step (§A + §B)

> "Adam; lr 10^-3 ... max epochs 30"
>
> "Monitor val Macro-F1; patience 7"
>
> — §A. Experimental Setup (Table I)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad(set_to_none=True)
logits = model(node_features=node_features, adj=adj, node_mask=node_mask)
loss = loss_fn(logits=logits, targets=targets)
loss.backward()
optimizer.step()

has_any_grad = False
for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        has_any_grad = True
        break
assert has_any_grad, 'No gradients found in model parameters.'
print(f'✓ One training step complete. Loss: {loss.item():.4f}')

## Common Pitfalls

1. **QuOp hyperparameters are underspecified**: Algorithm 1 requires `h` and `q`, but fixed values are not given.
2. **QWalkVec implementation details are partially specified**: coin/shift operators are referenced, but exact operator construction is not fully detailed in this paper.
3. **QPE anchor policy is unspecified**: anchor count is given (8), but node-selection strategy is not fixed.
4. **Macro metrics can diverge from accuracy**: the paper reports both; class imbalance can hide in accuracy-only checks.
5. **Exact paper scores depend on full data protocol**: splits, preprocessing parity, and external benchmark conventions matter.